In [1]:
using StatsBase
using JLD
using DataFrames
using Statistics
using Distributions
using Plots

import IterTools

include("model.jl")

get_sub_data (generic function with 1 method)

# Helper functions

In [2]:
# helper functions from get_best_fitting_parameters.ipynb
function get_block(true_planet)
    block =[]
    for p in true_planet
        if p < 20
            append!(block,1)
        elseif p < 40
            append!(block,2)
        elseif p < 60
            append!(block,3)
        elseif p < 80
            append!(block,4)
        elseif p < 100
            append!(block,5)
        end
    end
    return block
end


function chunk_galaries(galaxies)
    groups=[]
    for i in IterTools.groupby(x -> x, galaxies)
        push!(groups,i)
    end
        return groups
end

function number_planet_in_galaxy(galaxies)
    chunks = chunk_galaries(galaxies)
    return [i for galaxy in chunks for i in 1:length(galaxy)]
end

number_planet_in_galaxy (generic function with 1 method)

In [3]:
# function get_sub_data(sub_num)
#     # get the data of a specific subject with sub_num (sub id)
#     all_data=DataFrame(CSV.File("./data/all_data_dataset_2022.csv",delim=','))
#     sub_data = all_data[in(sub_num).(all_data.sub_num),:]
# end

In [4]:
function sample_planet_type(curr_planet_type, self_transition)
    # deciding what the next planet is based on self_transition
    # the p the next planet has the same glaxy type/planet type as the current one is self_transition
    # the p that the next planet has either of the others types is (1-self_transition)/2

    all_planets = [0,1,2]
    other_planets = filter(e->e≠curr_planet_type,all_planets)
    planets = vcat([curr_planet_type],other_planets)
    other_transition = (1-self_transition)/2
    weights=[self_transition,other_transition,other_transition]
    return sample(planets, Weights(weights))
end

function gen_triple_patch_exp_structure(n_blocks, n_planets_per_block, n_stays, p_self_trans, poor_decay_a, poor_decay_b, neutral_decay_a, neutral_decay_b, rich_decay_a, rich_decay_b, init_reward_mu, init_reward_sigma)
    # generating experiment structure for 3 patches environment (3 different galaxies)

    # initialize distributions for each galaxy type
    inital_reward_dist = Normal(init_reward_mu,init_reward_sigma)
    
    println(poor_decay_a, rich_decay_b)
    poor_decay_dist = Beta(poor_decay_a,poor_decay_b)
    neutral_decay_dist = Beta(neutral_decay_a,neutral_decay_b)
    rich_decay_dist = Beta(rich_decay_a,rich_decay_b)
    
    
    # intialize vectors to store data 
    store_block = []    # block number, [1, n_blocks]
    store_planet = []   # planet/true planet number, [0, (n_planets_per_block)*block number - 1]
    store_planet_in_block = []  # planet num per block, [0, n_planets_per_block-1]
    store_galaxy = []   # galaxy type/richness level, {0 (poor),1 (neutral),2 (rich)}
    store_stay_num = [] # stay number, it will go to the max since there is no time limit (time limit is imiplied in the model)
    store_reward = []   # the amount of reward
    true_decay = []
    for b in 1:n_blocks # did 1 indexing for blocks 
        # beginning of block
        # sample a planet type from a uniform distribution 
        all_planets = [0,1,2]
        current_planet_type = sample(all_planets, Weights([0.33,0.33,0.33])) # new planet for the first trial of each block
        for p in 0:(n_planets_per_block-1) # for some reason i did zero indexing for planets
            # sample an initial reward for the first dig on the planet  
            reward = round(rand(inital_reward_dist))
            decay = 0
            for stay_num = 0:(n_stays-1) # for some reason i did zero indexing for stay num 
                # save data 
                append!(store_block,b)
                append!(store_planet,p + (b-1)*n_planets_per_block) # for this true_planet and planet will be the same 
                append!(store_planet_in_block,p)
                append!(store_galaxy,current_planet_type)
                append!(store_stay_num,stay_num)
                append!(store_reward,reward)
                append!(true_decay, decay)
                
                # decay 
                if current_planet_type == 0
                    decay = rand(poor_decay_dist)
                    reward = reward*decay
                elseif current_planet_type == 1
                    decay = rand(neutral_decay_dist)
                    reward = reward*decay
                elseif current_planet_type == 2
                    decay = rand(rich_decay_dist)
                    reward = reward*decay
                end
            end
            # transition to a new planet 
            current_planet_type = sample_planet_type(current_planet_type,p_self_trans)
        end
    end
    # store everything in a data frame 
    exp_struc = DataFrame(Dict("block"=> store_block,"planet"=> store_planet,"planet_in_block"=> store_planet_in_block,"galaxy"=> store_galaxy,"stay_num"=> store_stay_num,"reward"=>store_reward, "true_decay"=>true_decay))
    return exp_struc 
end

# function gen_single_patch_exp_structure(n_blocks, n_planets_per_block, n_stays, decay_a, decay_b, init_reward_mu, init_reward_sigma)
#     # generating experiment structure for single patch environment

#     # initialize distributions 
#     inital_reward_dist = Normal(init_reward_mu, init_reward_sigma)
    
#     decay_dist = Beta(decay_a,decay_b)
    
#     # intialize vectors to store data 
#     store_block = []
#     store_planet = []
#     store_planet_in_block = []
#     store_stay_num = []
#     store_reward = []
    
#     for b in 1:n_blocks # did 1 indexing for blocks 
#         # beginning of block
#         # sample a planet type from a uniform distribution 
#         for p in 0:(n_planets_per_block-1) # for some reason i did zero indexing for planets
#             # sample an initial reward for the first dig on the planet  
#             reward = round(rand(inital_reward_dist))
#             for stay_num = 0:(n_stays-1) # for some reason i did zero indexing for stay num 
#                 # save data 
#                 append!(store_block,b)
#                 append!(store_planet,p + (b-1)*n_planets_per_block) # for this true_planet and planet will be the same 
#                 append!(store_planet_in_block,p) # for this true_planet and planet will be the same 
#                 append!(store_stay_num,stay_num)
#                 append!(store_reward,reward)
                
#                 reward = round(reward*rand(decay_dist))
#             end
#         end
#     end
#     # store everything in a data frame 
#     exp_struc = DataFrame(Dict("block"=> store_block,"planet"=> store_planet,"planet_in_block"=> store_planet_in_block,"stay_num"=> store_stay_num,"reward"=>store_reward))
#     return exp_struc 
# end

gen_triple_patch_exp_structure (generic function with 1 method)

In [5]:
function gen_triple_patch_exp_structure_predefined(n_blocks, n_planets_per_block, n_stays, planet_lst, poor_decay_a, poor_decay_b, neutral_decay_a, neutral_decay_b, rich_decay_a, rich_decay_b, init_reward_mu, init_reward_sigma)
    # generating experiment structure for 3 patches environment (3 different galaxies)

    # initialize distributions for each galaxy type
    inital_reward_dist = Normal(init_reward_mu,init_reward_sigma)
    
    println(poor_decay_a, rich_decay_b)
    poor_decay_dist = Beta(poor_decay_a,poor_decay_b)
    neutral_decay_dist = Beta(neutral_decay_a,neutral_decay_b)
    rich_decay_dist = Beta(rich_decay_a,rich_decay_b)
    
    
    # intialize vectors to store data 
    store_block = []    # block number, [1, n_blocks]
    store_planet = []   # planet/true planet number, [0, (n_planets_per_block)*block number - 1]
    store_planet_in_block = []  # planet num per block, [0, n_planets_per_block-1]
    store_galaxy = []   # galaxy type/richness level, {0 (poor),1 (neutral),2 (rich)}
    store_stay_num = [] # stay number, it will go to the max since there is no time limit (time limit is imiplied in the model)
    store_reward = []   # the amount of reward
    true_decay = []
    for b in 1:n_blocks # did 1 indexing for blocks 
        # beginning of block
        # sample a planet type from a uniform distribution 
        all_planets = [0,1,2]
        current_planet_type = planet_lst[b][1] # new planet for the first trial of each block
        for p in 0:(n_planets_per_block-1) # for some reason i did zero indexing for planets
            # sample an initial reward for the first dig on the planet  
            reward = round(rand(inital_reward_dist))
            decay = 0
            for stay_num = 0:(n_stays-1) # for some reason i did zero indexing for stay num 
                # save data 
                append!(store_block,b)
                append!(store_planet,p + (b-1)*n_planets_per_block) # for this true_planet and planet will be the same 
                append!(store_planet_in_block,p)
                append!(store_galaxy,current_planet_type)
                append!(store_stay_num,stay_num)
                append!(store_reward,reward)
                append!(true_decay, decay)
                
                # decay 
                if current_planet_type == 0
                    decay = rand(poor_decay_dist)
                    reward = reward*decay
                elseif current_planet_type == 1
                    decay = rand(neutral_decay_dist)
                    reward = reward*decay
                elseif current_planet_type == 2
                    decay = rand(rich_decay_dist)
                    reward = reward*decay
                end
            end
            # transition to a new planet 
            if p!= (n_planets_per_block-1)
                current_planet_type = planet_lst[b][p+2]
            end
        end
    end
    # store everything in a data frame 
    exp_struc = DataFrame(Dict("block"=> store_block,"planet"=> store_planet,"planet_in_block"=> store_planet_in_block,"galaxy"=> store_galaxy,"stay_num"=> store_stay_num,"reward"=>store_reward, "true_decay"=>true_decay))
    return exp_struc 
end

gen_triple_patch_exp_structure_predefined (generic function with 1 method)

In [6]:
function gen_exp_struc(p_self_trans, decay_var, beta_paras, planet_lst=-1, n_blocks=5, n_planets_per_block=20, n_stays_max=20, init_reward_mu=100, init_reward_sigma=5, ver=1)
    # generating experiment struc with maximum nubmer of possible stay
    poor_decay_a = beta_paras[1]
    poor_decay_b = beta_paras[2]
    neutral_decay_a = beta_paras[3]
    neutral_decay_b = beta_paras[4]
    rich_decay_a = beta_paras[5]
    rich_decay_b = beta_paras[6]

    if isa(planet_lst, AbstractArray)
        exp_struc = gen_triple_patch_exp_structure_predefined(n_blocks, n_planets_per_block, n_stays_max, planet_lst, poor_decay_a, poor_decay_b, neutral_decay_a, neutral_decay_b, rich_decay_a, rich_decay_b, init_reward_mu, init_reward_sigma)
        if ver==1
            CSV.write(string("exp_struc/","pre_exp_struc_", Int(p_self_trans*100), "_", Int(decay_var*100), ".csv"), exp_struc)
        else
            CSV.write(string("exp_struc/","pre_exp_struc2_", Int(p_self_trans*100), "_", Int(decay_var*100), ".csv"), exp_struc)
        end
    else
        exp_struc = gen_triple_patch_exp_structure(n_blocks, n_planets_per_block, n_stays_max, p_self_trans, poor_decay_a, poor_decay_b, neutral_decay_a, neutral_decay_b, rich_decay_a, rich_decay_b, init_reward_mu, init_reward_sigma)
        if ver==1
            CSV.write(string("exp_struc/","exp_struc_", Int(p_self_trans*100), "_", Int(decay_var*100), ".csv"), exp_struc)
        else
            CSV.write(string("exp_struc/","exp_struc2_", Int(p_self_trans*100), "_", Int(decay_var*100), ".csv"), exp_struc)
        end
    end
    
    return exp_struc
end 

gen_exp_struc (generic function with 8 methods)

In [7]:
function simulations_some_paras(exp_struc, filename, beta_paras, paras_lst=[1,5,2], n_sim=1)
    # generate n_sim simulations based on exp_struc, save the csv named as filename
    # paras = [alpha, gamma_base, gamma_coef]
    sims = DataFrame(sim_num=Int[], trial=Int[], true_planet=Int[], planet=Int[], galaxy=Int[], stay_num=Float64[], reward=Float64[], entropy=Float64[])
    decay_lst = []
    galaxy_lst = []
    r_lst = []

    sim_num = 1

    for subj in 1:1
        # alpha, gamma_base, gamma_coef
        paras = paras_lst
        for i in 1:n_sim
            data = copy(exp_struc)
            num_particles = 1

            # the model, implying time limit
            # b -> behavior: including a few lists: true_planet, prt, glaxy, reward_list, in by planet style (you can stay more than 1 time in each planet, resulting in multiple trials on one planet)
            # the rest -> by trial style
            b, true_planet_lst, planet_lst, true_galaxy_lst, stay_num_lst, reward_lst, entropy_lst=crp_adaptiveDiscount(data,paras,num_particles,beta_paras)

            # # use optimal policy for multi patches inference (participants think there isn't just one group)
            # opt_prt = optimal_policy(b) # optimal prt

            # # previous df that will return by planet df
            # # diff between actual and optimal prt
            # diff = b.prt - opt_prt
            # # simple ver
            # df = DataFrame(Dict("true_planet"=> b.true_planet,"galaxy"=> b.galaxy,"prt"=>b.prt,
            #     "opt_prt"=>opt_prt, "diff" => diff))

            # # gdf = groupby(df, :galaxy)
            # # prt_avg = combine(gdf, :diff => mean)
            
            # # complicated one: to use this one, you need to change the crp_adaptiveDiscount below
            # df = DataFrame(Dict("sim_num"=>sim_num, "true_planet"=> b.true_planet,"galaxy"=> b.galaxy,"prt"=>b.prt,
            #     "opt_prt"=>opt_prt, "diff" => diff,"pred_decay"=>x,"uncertainty"=>y,"true_decay"=>z, "entropy"=>m_u, 
            #     "alpha"=>paras[1], "gamma_base"=>paras[2], "gamma_coef"=>paras[3]))

            # new by trial df
            df = DataFrame(Dict("sim_num"=>sim_num, "trial"=>1:length(true_planet_lst), "true_planet"=> true_planet_lst,"planet"=> planet_lst, "galaxy"=> true_galaxy_lst,"stay_num"=>stay_num_lst,
                "reward"=>reward_lst, "entropy" => entropy_lst))
            
            sims = vcat(sims, df)
            
            sim_num = sim_num + 1
        end
    end

    CSV.write(string("sims/", filename),sims)
    # println(r_lst)
    # println(decay_lst)
    return sims
end

simulations_some_paras (generic function with 3 methods)

# sim some paras

In [12]:
function crp_adaptiveDiscount(exp_struc,params,num_particles,beta_paras)
    # intialize the exp
    expt = Experiment(5,360,20,2,10,5.5,1.5) # change needed if we want to manipulate the time
    behav = Behavior([],[],[],[])   # true_planet, prt, galaxy, reward_lst

    # initialize the particle filter
    alpha = params[1]
    gamma_base = params[2]
    gamma_coef = params[3]
    #println(params)
    hyper_mu = 0.5
    hyper_var = 0.25
    hyper_tau = 1
    n_samples = 100
    particles, weights = init_particle_filter(num_particles,hyper_mu, hyper_var, hyper_tau, alpha)

    # initialize for tracking progress through the exp
    total_reward = 0
    total_time = expt.alien_time + expt.iti
    N = 0 # all planets experienced, ***check if changes if I change to number of decay rates ***
    n_cluster_inferred = []
    gammas = []
    pred_decay = []
    uncertainty=[]
    true_decay=[]
    middle = []
    middle_decay = []
    true_decay_lst = []

    # YC added 06-26-2024
    true_galaxy_lst = []
    true_planet_lst = [] # YC
    planet_lst = [] # YC
    current_planet_num = 0
    stay_num_lst = [] # YC
    reward_lst = []
    entropy_lst = []  # YC added, 11-30-23


    for b in 1:expt.n_blocks    # go throught each block?
        curr_block = exp_struc[in(b).(exp_struc.block),:]   # data for the current block
        b_tracker = Block_Tracker(expt.alien_time,0) # block time, planet (current planet num?)
        while (b_tracker.block_time < expt.block_max_time) & (b_tracker.planet < expt.block_n_planet) # while within time limit + do NOT exceed the max planet num (20 for now)
            curr_planet, p_tracker = get_planet(b, b_tracker,curr_block)

            while (b_tracker.block_time < expt.block_max_time) & p_tracker.on_planet

                # dig up reward and add to list
                p_tracker, total_reward = get_reward(p_tracker, total_reward, curr_planet)

                # add time that it took to dig it up
                b_tracker.block_time += expt.harvest_time + expt.iti
                total_time += expt.harvest_time + expt.iti

                # probability of cluster
                global prob_k = prob_cluster(p_tracker.curr_reward, p_tracker.decay_list, N, alpha, particles)
                samples,clusters = sample_v_stay(particles,weights, prob_k, n_samples,p_tracker.decay_list)
                global samples_all = samples

                clusters = round.(Int, clusters)
                K = maximum(clusters)
                append!(n_cluster_inferred,length(prob_k[1]))
                d =Multinomial(n_samples,tally(clusters,K)/n_samples)
                model_uncertainty = entropy(d)
                # println(model_uncertainty)
                gamma_effective = 1/(1+exp(-(gamma_base + gamma_coef*model_uncertainty)))   # model uncertainty is H(X)
                v_stay = mean(samples)*p_tracker.curr_reward
                v_leave = (total_reward/total_time)*(expt.harvest_time + expt.iti)*gamma_effective
                p_tracker =  make_choice_max(v_stay, v_leave, p_tracker)
                append!(middle,[tally(clusters,K)])
                append!(middle_decay,[deepcopy(p_tracker.decay_list)])
                append!(reward_lst, p_tracker.curr_reward)  # YC added, 11-23
                append!(entropy_lst, model_uncertainty)
                append!(true_galaxy_lst, p_tracker.galaxy)
                append!(true_planet_lst, p_tracker.true_planet)
                append!(stay_num_lst, p_tracker.prt)

                if p_tracker.prt==1 && length(planet_lst)>0
                    current_planet_num = current_planet_num+1
                end 

                append!(planet_lst, current_planet_num)
            end
            append!(uncertainty,var(samples_all))
            append!(pred_decay,mean(samples_all))
            append!(middle,[[]])
            append!(middle_decay,[[]])

            if p_tracker.prt > 1
                particles, weights = resample_and_update_particles(particles,weights,prob_k,p_tracker.decay_list)
            end

            if p_tracker.prt > 1
                append!(true_decay,mean(p_tracker.decay_list))
                append!(true_decay_lst, p_tracker.decay_list)
                # append!(true_galaxy_lst, repeat([p_tracker.galaxy], length(p_tracker.decay_list)))
                for idx in 2:length(p_tracker.reward_list)
                    dd = round(p_tracker.decay_list[idx-1], digits=3)
                    old_r = p_tracker.reward_list[idx-1]
                    new_r = p_tracker.reward_list[idx]
                    actual_dd = round(new_r/old_r, digits=3)
                    if abs(actual_dd-dd)>0.1
                        println("!!!")
                        println(dd, " ", actual_dd)
                    end
                end
            else
                append!(true_decay,0)
            end
            b_tracker.block_time += expt.travel_time + expt.alien_time
            b_tracker.planet += 1
            total_time += expt.travel_time + expt.alien_time
            behav = update_behavior(behav, p_tracker)
            N += 1
            # println(N-1, " DONE, on block ", b, " galaxy ", behav.galaxy)

        end
    end
    true_decay_lst = filter(x -> !isnan(x), true_decay_lst)
    true_galaxy_lst = filter(x -> !isnan(x), true_galaxy_lst)
    # println(mean(vars)^0.5)
    return behav, true_planet_lst, planet_lst, true_galaxy_lst, stay_num_lst, reward_lst, entropy_lst
end

crp_adaptiveDiscount (generic function with 2 methods)

In [9]:
beta_paras_5 = [13, 51, 50, 50, 50, 12]
struc_80 = [[0, 0, 1, 1, 1, 2, 2, 2, 0, 0, 0, 0, 0, 2, 2, 1, 2, 2, 2, 2], 
       [0, 0, 0, 2, 2, 2, 2, 2, 1, 2, 2, 0, 0, 0, 0, 2, 2, 2, 2, 2], 
       [1, 2, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2], 
       [0, 0, 0, 2, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2], 
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1]]
       
exp_80_5_predefined = gen_exp_struc(0.8, 0.05, beta_paras_5, struc_80)

1312


Row,block,galaxy,planet,planet_in_block,reward,stay_num,true_decay
,Any,Any,Any,Any,Any,Any,Any
1,1,0,0,0,98.0,0,0
2,1,0,0,0,19.2499,1,0.196428
3,1,0,0,0,3.21086,2,0.166799
4,1,0,0,0,0.67954,3,0.211638
5,1,0,0,0,0.0948648,4,0.139601
6,1,0,0,0,0.0245496,5,0.258785
7,1,0,0,0,0.00640403,6,0.260861
8,1,0,0,0,0.000950232,7,0.14838
9,1,0,0,0,0.000273326,8,0.287642


In [10]:
behav, true_planet_lst, planet_lst, true_galaxy_lst, stay_num_lst, reward_lst, entropy_lst = crp_adaptiveDiscount(exp_80_5_predefined,[3,0.1,0.2],1,beta_paras_5)

MethodError: MethodError: no method matching crp_adaptiveDiscount(::DataFrame, ::Vector{Float64}, ::Int64, ::Vector{Int64})

Closest candidates are:
  crp_adaptiveDiscount(::Any, ::Any, ::Any)
   @ Main ~/Desktop/foraging_MQLP/model/model.jl:207


In [75]:
planet_lst

277-element Vector{Any}:
  0
  0
  1
  1
  1
  2
  2
  2
  2
  3
  ⋮
 52
 52
 52
 53
 53
 53
 54
 54
 54

In [74]:
true_planet_lst

277-element Vector{Any}:
  0
  0
  1
  1
  1
  2
  2
  2
  2
  3
  ⋮
 90
 90
 90
 91
 91
 91
 92
 92
 92

In [73]:
true_galaxy_lst

277-element Vector{Any}:
 0
 0
 0
 0
 0
 1
 1
 1
 1
 1
 ⋮
 1
 1
 1
 1
 1
 1
 1
 1
 1

In [76]:
stay_num

277-element Vector{Any}:
 1
 2
 1
 2
 3
 1
 2
 3
 4
 1
 ⋮
 1
 2
 3
 1
 2
 3
 1
 2
 3

In [77]:
rewards

277-element Vector{Any}:
  97.0
  19.663195543117833
 101.0
  29.323942923058517
   4.635332843952455
 101.0
  49.78476090209976
  22.744635370919745
  11.747108441396527
 103.0
   ⋮
 104.0
  44.62402641489853
  23.206363344782904
 106.0
  50.110023647357295
  26.145402537828737
 102.0
  50.12666697355811
  24.334146363719203

In [78]:
entropy_lst

277-element Vector{Any}:
 0.0
 0.0
 2.7397003984594304
 3.003389442393295
 2.998496969559644
 3.011785674753526
 3.027561846495452
 2.980875370018269
 2.980875370018269
 3.018402574474706
 ⋮
 6.520661677632178
 1.6978473556803912
 0.0
 6.515467114491145
 3.907171680154419
 1.6978473556803912
 6.446967895325563
 3.2215169713388407
 1.3024421011717318

In [79]:
rewards

277-element Vector{Any}:
  97.0
  19.663195543117833
 101.0
  29.323942923058517
   4.635332843952455
 101.0
  49.78476090209976
  22.744635370919745
  11.747108441396527
 103.0
   ⋮
 104.0
  44.62402641489853
  23.206363344782904
 106.0
  50.110023647357295
  26.145402537828737
 102.0
  50.12666697355811
  24.334146363719203

In [13]:
# alpha, gamma_base, gamma_coef
alpha_0_df = simulations_some_paras(exp_80_5_predefined, "alpha_0_sims.csv", beta_paras_5, [0,0.1,0.2], 100)
alpha_1_df = simulations_some_paras(exp_80_5_predefined, "alpha_1_sims.csv", beta_paras_5, [1,0.1,0.2], 100)
alpha_5_df = simulations_some_paras(exp_80_5_predefined, "alpha_5_sims.csv", beta_paras_5, [5,0.1,0.2], 100)
alpha_10_df = simulations_some_paras(exp_80_5_predefined, "alpha_10_sims.csv", beta_paras_5, [10,0.1,0.2], 100)

Row,sim_num,trial,true_planet,planet,galaxy,stay_num,reward,entropy
,Int64,Int64,Any,Any,Any,Any,Any,Any
1,1,1,0,0,0,1,98.0,0.0
2,1,2,0,0,0,2,19.2499,0.0
3,1,3,1,1,0,1,106.0,2.34519
4,1,4,1,1,0,2,33.1589,2.51123
5,1,5,1,1,0,3,6.52,2.55413
6,1,6,2,2,1,1,105.0,4.33879
7,1,7,2,2,1,2,47.885,4.7502
8,1,8,2,2,1,3,21.9522,5.38197
9,1,9,3,3,1,1,107.0,7.18584


In [20]:
alpha_1_df

Row,sim_num,trial,true_planet,planet,galaxy,stay_num,reward,entropy
,Int64,Int64,Any,Any,Any,Any,Any,Any
1,1,1,0,0,0,1,107.0,0.0
2,1,2,0,0,0,2,25.7915,0.0
3,1,3,1,1,0,1,99.0,3.02756
4,1,4,1,1,0,2,19.6295,3.02107
5,1,5,2,2,1,1,112.0,2.91996
6,1,6,2,2,1,2,67.9458,2.99312
7,1,7,2,2,1,3,38.3826,2.98088
8,1,8,2,2,1,4,18.2652,2.78363
9,1,9,3,3,1,1,101.0,5.7125


In [18]:
# high_a_df = simulations_some_paras(exp_80_5_predefined, "high_a.csv", beta_paras_5, [3,0.1,0.2])
# mid_a_df = simulations_some_paras(exp_80_5_predefined, "mid_a.csv", beta_paras_5, [1,0.1,0.2])
# low_a_df = simulations_some_paras(exp_80_5_predefined, "low_a.csv", beta_paras_5, [0,0.1,0.2])

## check dist

In [67]:
# check_decay_dist(decay_50_5, galaxy_50_5)
# check_decay_dist(decay_70_5, galaxy_70_5)
# check_decay_dist(decay_80_5, galaxy_80_5)

check_decay_dist(decay_60_5_predefined, galaxy_60_5_predefined)
check_decay_dist(decay_70_5_predefined, galaxy_70_5_predefined)
check_decay_dist(decay_80_5_predefined, galaxy_80_5_predefined)

0 46
1 66
2 115
galaxy==0 0.19778510133064256 0.04368984435018343
galaxy==1 0.4977102051303975 0.04620278609402704
galaxy==2 0.8147595384690163 0.05139348895821262
0 51
1 81
2 57
galaxy==0 0.20250731081516507 0.05328208658969906
galaxy==1 0.503260625243519 0.04536815261645212
galaxy==2 0.8172441156432008 0.04964043349787855
0 27
1 65
2 111
galaxy==0 0.20726394435192178 0.061897493226581744
galaxy==1 0.4973999480670418 0.044630902674399085
galaxy==2 0.807773933538037 0.05073154187646134


In [68]:
# check_decay_dist(decay_50_8, galaxy_50_8)
# check_decay_dist(decay_70_8, galaxy_70_8)
# check_decay_dist(decay_80_8, galaxy_80_8)

check_decay_dist(decay_60_8_predefined, galaxy_60_8_predefined)
check_decay_dist(decay_70_8_predefined, galaxy_70_8_predefined)
check_decay_dist(decay_80_8_predefined, galaxy_80_8_predefined)

0 49
1 57
2 97
galaxy==0 0.23477303988644768 0.0793674350871528
galaxy==1 0.49882893659475974 0.06403935536409548
galaxy==2 0.7792916573868621 0.07876766259017794
0 63
1 84
2 39
galaxy==0 0.21851354154409336 0.07956431349532603
galaxy==1 0.48730679519392966 0.07279286376254715
galaxy==2 0.7719104514200371 0.06346309998791702
0 25
1 77
2 101
galaxy==0 0.221646635942875 0.0750097815855965
galaxy==1 0.47983229236974156 0.0708686540738857
galaxy==2 0.7461662965892614 0.08636577538825063


In [69]:
# check_decay_dist(decay_50_10, galaxy_50_10)
# check_decay_dist(decay_70_10, galaxy_70_10)
# check_decay_dist(decay_80_10, galaxy_80_10)

check_decay_dist(decay_60_10_predefined, galaxy_60_10_predefined)
check_decay_dist(decay_70_10_predefined, galaxy_70_10_predefined)
check_decay_dist(decay_80_10_predefined, galaxy_80_10_predefined)

0 52
1 66
2 102
galaxy==0 0.23475085612095203 0.1006197187077716
galaxy==1 0.4949142530881215 0.09473684957833733
galaxy==2 0.768247083999934 0.10060038191709841
0 61
1 74
2 38
galaxy==0 0.24749481595548453 0.0968651514456208
galaxy==1 0.514520913621872 0.09260148448616791
galaxy==2 0.7520701896596615 0.08932330235795442
0 29
1 85
2 88
galaxy==0 0.22667771324824748 0.11120844544523736
galaxy==1 0.5211715924199759 0.09396770846936721
galaxy==2 0.7654189548505165 0.09609418923619256
